In [1]:
#!pip install pandas numpy matplotlib seaborn scikit-learn statsmodels

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from statsmodels.formula.api import ols
import warnings
warnings.filterwarnings('ignore')

# Set plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [3]:
# Load the data
excel_file = 'SOBSS Branded PS Data 2023.xlsx'

# Inspect available sheets
xls = pd.ExcelFile(excel_file)
print("Available sheets:")
print(xls.sheet_names)
print("\n")

# Read all data sheets (exclude the Data Dictionary)
data_sheets = [s for s in xls.sheet_names if s.lower().strip() != 'data dictionary']
print(f"Loading sheets: {data_sheets}")

df_list = []
for s in data_sheets:
    df_tmp = pd.read_excel(excel_file, sheet_name=s)
    df_tmp['source_sheet'] = s
    df_list.append(df_tmp)

# Concatenate all region/channel sheets
df_raw = pd.concat(df_list, ignore_index=True)
print(f"Combined raw shape: {df_raw.shape}")



Available sheets:
['Data Dictionary', 'US SEO', 'US PPC', 'Canada PPC', 'Canada SEO']


Loading sheets: ['US SEO', 'US PPC', 'Canada PPC', 'Canada SEO']
Combined raw shape: (13365, 14)


In [4]:
# Normalize some column names for easier handling
rename_map = {
    'BRAND_ads_ON_Flag (valid during test period only)': 'BRAND_ads_ON_Flag',
    'Hour Designation (during test period)': 'Hour_Designation',
    'Test Preiod (1 = Test Period, 0 = Non-test period)': 'Test_Period'
}

df_raw = df_raw.rename(columns=rename_map)

df_raw = df_raw.drop(columns= ['Hour_Designation','cnt_obsrv'])

print('\nColumns after rename:')
print(df_raw.columns.tolist())




Columns after rename:
['Country', 'Channel', 'year', 'month', 'day', 'Hour', 'Test_Period', 'BRAND_ads_ON_Flag', 'Users', 'NewUsers', 'Trials', 'source_sheet']


In [5]:
# Ensure required numeric columns exist
for col in ['Users', 'NewUsers', 'Trials']:
    if col not in df_raw.columns:
        raise KeyError(f"Required column '{col}' not found in data sheets")
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

# Create separate DataFrames per source sheet
# Keep Hour column (do not drop it)
df_us_seo = df_raw[df_raw['source_sheet'] == 'US SEO'].copy()
df_us_ppc = df_raw[df_raw['source_sheet'] == 'US PPC'].copy()
df_ca_ppc = df_raw[df_raw['source_sheet'] == 'Canada PPC'].copy()
df_ca_seo = df_raw[df_raw['source_sheet'] == 'Canada SEO'].copy()

print(f"US SEO shape: {df_us_seo.shape}")
print(f"US PPC shape: {df_us_ppc.shape}")
print(f"Canada PPC shape: {df_ca_ppc.shape}")
print(f"Canada SEO shape: {df_ca_seo.shape}")



US SEO shape: (3453, 12)
US PPC shape: (3410, 12)
Canada PPC shape: (3157, 12)
Canada SEO shape: (3345, 12)


In [6]:
# Aggregation keys (per country totals)
agg_keys = ['year', 'month', 'day', 'Hour', 'BRAND_ads_ON_Flag']
for k in agg_keys:
    if k not in df_raw.columns:
        raise KeyError(f"Missing aggregation key: {k}")

# Total for United States
df_total_US = (
    df_raw[df_raw['Country'].str.lower().str.contains('united')]
    .groupby(agg_keys)[['Users', 'NewUsers', 'Trials']]
    .sum()
    .reset_index()
    .rename(columns={'Users': 'total_users_US', 'NewUsers_US': 'total_newusers_US', 'Trials': 'total_trials_US'})
)

# Total for Canada
df_total_Canada = (
    df_raw[df_raw['Country'].str.lower().str.contains('canada')]
    .groupby(agg_keys)[['Users', 'NewUsers', 'Trials']]
    .sum()
    .reset_index()
    .rename(columns={'Users': 'total_users_CA', 'NewUsers': 'total_newusers_CA', 'Trials': 'total_trials_CA'})
)

print(f"\nTotal US aggregated shape: {df_total_US.shape}")
print(f"Total Canada aggregated shape: {df_total_Canada.shape}")

# Show a few rows of each aggregated DF
print('\nSample df_total_US:')
print(df_total_US.head())
print('\nSample df_total_Canada:')
print(df_total_Canada.head())





Total US aggregated shape: (3456, 8)
Total Canada aggregated shape: (3409, 8)

Sample df_total_US:
   year  month  day  Hour  BRAND_ads_ON_Flag  total_users_US  NewUsers  \
0  2023      5    1     0                  0              21         7   
1  2023      5    1     1                  0              20         8   
2  2023      5    1     2                  1              17         8   
3  2023      5    1     3                  1              11         5   
4  2023      5    1     4                  0               4         2   

   total_trials_US  
0                0  
1                1  
2                1  
3                0  
4                0  

Sample df_total_Canada:
   year  month  day  Hour  BRAND_ads_ON_Flag  total_users_CA  \
0  2023      5    1     0                  0              16   
1  2023      5    1     1                  0               9   
2  2023      5    1     2                  1               4   
3  2023      5    1     3                  1    

In [7]:
df_total_Canada.head()

,year,month,day,Hour,BRAND_ads_ON_Flag,total_users_CA,total_newusers_CA,total_trials_CA
0,2023,5,1,0,0,16,5,1
1,2023,5,1,1,0,9,3,2
2,2023,5,1,2,1,4,1,2
3,2023,5,1,3,1,4,1,0
4,2023,5,1,4,0,1,0,0


In [8]:
df_us_seo.head()

,Country,Channel,year,month,day,Hour,Test_Period,BRAND_ads_ON_Flag,Users,NewUsers,Trials,source_sheet
0,United States,SEO,2023,5,1,0,0,0,12,3,0,US SEO
1,United States,SEO,2023,5,1,1,0,0,12,4,0,US SEO
2,United States,SEO,2023,5,1,2,0,1,9,1,0,US SEO
3,United States,SEO,2023,5,1,3,0,1,6,2,0,US SEO
4,United States,SEO,2023,5,1,4,0,0,3,2,0,US SEO
